# TEP reactor-pressure candidate study

Generated from `tep_researcher_input.ipynb`. This notebook implements the authored mechanics without adding persistence, merging, or completeness rules. Its objects are **candidate intervals**, not validated complete physical episodes.

In [ ]:
import featuregraph as fg
import pandas as pd

FAULT_NUMBER = 2
SIMULATION_RUN = 10
SIGNAL = "reactor_pressure"
ROLLING_MAX_WINDOW = 50
ROLLING_MEAN_WINDOW = 50
OFFLINE_ALIGNMENT_SHIFT = 50
RATE_EPS = 0.0


def load_run(fault_number=FAULT_NUMBER, simulation_run=SIMULATION_RUN):
    """Load one Tennessee Eastman Process run without altering source values."""
    return fg.datasets.eastman(
        fault_number=fault_number,
        simulation_run=simulation_run,
    ).copy()


def construct_observations(source, rate_eps=RATE_EPS):
    """Apply exactly the preprocessing, state, event, and identity rules."""
    df = source.copy()
    original_pressure = df[SIGNAL].copy(deep=True)

    df["sample_index"] = df.index
    df["time_hours"] = df["time_(h)"]
    df["reactor_pressure_raw"] = df[SIGNAL]

    time_step_hours = df["time_hours"].diff()
    sample_interval_minutes = time_step_hours.median() * 60
    if not pd.notna(sample_interval_minutes) or sample_interval_minutes <= 0:
        raise ValueError("The inferred sample interval must be finite and positive.")
    if not time_step_hours.dropna().gt(0).all():
        raise ValueError("time_hours must be strictly increasing.")

    df["reactor_pressure_smooth"] = (
        df["reactor_pressure_raw"]
        .rolling(ROLLING_MAX_WINDOW, min_periods=ROLLING_MAX_WINDOW).max()
        .rolling(ROLLING_MEAN_WINDOW, min_periods=ROLLING_MEAN_WINDOW).mean()
        .shift(-OFFLINE_ALIGNMENT_SHIFT)
    )
    df["reactor_pressure_change"] = df["reactor_pressure_smooth"].diff()
    df["reactor_pressure_rate"] = (
        df["reactor_pressure_change"] / df["time_hours"].diff()
    )
    df["reactor_pressure_valid"] = (
        df["reactor_pressure_smooth"].notna()
        & df["reactor_pressure_rate"].notna()
    )

    valid = df["reactor_pressure_valid"]
    df["reactor_pressure_rising"] = (
        valid & df["reactor_pressure_rate"].gt(rate_eps)
    )
    df["reactor_pressure_falling"] = (
        valid & df["reactor_pressure_rate"].lt(-rate_eps)
    )
    df["reactor_pressure_inactive"] = (
        valid & df["reactor_pressure_rate"].abs().le(rate_eps)
    )

    rising_int = df["reactor_pressure_rising"].astype(int)
    transition_valid = valid & valid.shift(1, fill_value=False)
    df["reactor_pressure_transition_valid"] = transition_valid
    df["enter_reactor_pressure_rising"] = (
        transition_valid & rising_int.diff().eq(1)
    )
    df["exit_reactor_pressure_rising"] = (
        transition_valid & rising_int.diff().eq(-1)
    )
    df["reactor_pressure_peak"] = df["exit_reactor_pressure_rising"]
    df["reactor_pressure_cycle_id"] = (
        df["exit_reactor_pressure_rising"].cumsum()
    )

    provenance = {
        "fault_number": FAULT_NUMBER,
        "simulation_run": SIMULATION_RUN,
        "signal": SIGNAL,
        "sample_interval_minutes": float(sample_interval_minutes),
        "rolling_max_window_samples": ROLLING_MAX_WINDOW,
        "rolling_mean_window_samples": ROLLING_MEAN_WINDOW,
        "offline_alignment_shift_samples": -OFFLINE_ALIGNMENT_SHIFT,
        "rate_eps_pressure_units_per_hour": rate_eps,
    }
    return df, original_pressure, provenance


def summarize_candidates(df):
    """Construct half-open peak-to-peak cycles and retain fragments."""
    summary = (
        df.groupby("reactor_pressure_cycle_id", sort=True)
        .agg(
            start_index=("sample_index", "min"),
            end_index=("sample_index", "max"),
            start_time_hours=("time_hours", "min"),
            end_time_hours=("time_hours", "max"),
            reactor_pressure_rising_samples=("reactor_pressure_rising", "sum"),
            reactor_pressure_falling_samples=("reactor_pressure_falling", "sum"),
            reactor_pressure_inactive_samples=("reactor_pressure_inactive", "sum"),
            enter_rising_count=("enter_reactor_pressure_rising", "sum"),
            exit_rising_count=("exit_reactor_pressure_rising", "sum"),
            raw_minimum=("reactor_pressure_raw", "min"),
            raw_maximum=("reactor_pressure_raw", "max"),
            smooth_minimum=("reactor_pressure_smooth", "min"),
            smooth_maximum=("reactor_pressure_smooth", "max"),
            observation_count=("sample_index", "size"),
        )
        .reset_index()
        .rename(columns={"reactor_pressure_cycle_id": "candidate_id"})
    )
    peak_rows = df.loc[df["reactor_pressure_peak"], [
        "reactor_pressure_cycle_id", "sample_index", "time_hours",
        "reactor_pressure_raw", "reactor_pressure_smooth",
    ]].copy()
    peak_rows = peak_rows.rename(columns={
        "reactor_pressure_cycle_id": "candidate_id",
        "sample_index": "peak_index",
        "time_hours": "peak_time_hours",
        "reactor_pressure_raw": "peak_raw_pressure",
        "reactor_pressure_smooth": "peak_smooth_pressure",
    })
    peak_rows["next_peak_index"] = peak_rows["peak_index"].shift(-1)
    peak_rows["next_peak_time_hours"] = peak_rows["peak_time_hours"].shift(-1)
    summary = summary.merge(peak_rows, on="candidate_id", how="left", validate="one_to_one")
    summary["duration_hours"] = summary["next_peak_time_hours"] - summary["peak_time_hours"]
    summary["candidate_label"] = summary["candidate_id"].map(
        lambda value: f"TEP-{SIMULATION_RUN:02d}-{int(value):03d}"
    )
    summary["is_complete"] = summary["peak_index"].notna() & summary["next_peak_index"].notna()
    summary["boundary_fragment"] = ~summary["is_complete"]
    summary["scientific_status"] = summary["is_complete"].map({True: "complete_peak_to_peak_cycle", False: "boundary_fragment"})
    return summary


def validate_study(source, df, original_pressure, summary):
    """Return explicit structural checks; raise if any required check fails."""
    valid = df["reactor_pressure_valid"]
    state_count = df[
        [
            "reactor_pressure_rising",
            "reactor_pressure_falling",
            "reactor_pressure_inactive",
        ]
    ].sum(axis=1)
    rising_int = df["reactor_pressure_rising"].astype(int)
    transition_valid = df["reactor_pressure_transition_valid"]

    checks = {
        "time_strictly_increasing": df["time_hours"].diff().dropna().gt(0).all(),
        "raw_pressure_preserved": source[SIGNAL].equals(original_pressure),
        "one_state_per_valid_sample": state_count[valid].eq(1).all(),
        "no_state_on_invalid_sample": state_count[~valid].eq(0).all(),
        "enter_events_match_definition": df[
            "enter_reactor_pressure_rising"
        ].equals(transition_valid & rising_int.diff().eq(1)),
        "exit_events_match_definition": df[
            "exit_reactor_pressure_rising"
        ].equals(transition_valid & rising_int.diff().eq(-1)),
        "no_events_on_invalid_transitions": not df.loc[~transition_valid, ["enter_reactor_pressure_rising", "exit_reactor_pressure_rising"]].any(axis=None),
        "candidate_rows_cover_source": int(summary["observation_count"].sum()) == len(df),
        "boundary_fragments_retained": (
            summary.empty or int(summary["boundary_fragment"].sum()) in (1, 2)
        ),
    }
    report = pd.Series(checks, name="passed").rename_axis("check").to_frame()
    failed = report.index[~report["passed"]].tolist()
    if failed:
        raise AssertionError(f"Study validation failed: {failed}")
    return report


In [ ]:
source = load_run()
observations, original_pressure, provenance = construct_observations(source)
candidate_summary = summarize_candidates(observations)
validation_report = validate_study(
    source,
    observations,
    original_pressure,
    candidate_summary,
)

provenance


In [ ]:
fragmentation_diagnostics = pd.Series(
    {
        "source_observations": len(observations),
        "valid_observations": int(observations["reactor_pressure_valid"].sum()),
        "all_intervals": len(candidate_summary),
        "complete_peak_to_peak_cycles": int(candidate_summary["is_complete"].sum()),
        "enter_rising_events": int(
            observations["enter_reactor_pressure_rising"].sum()
        ),
        "peak_events": int(
            observations["exit_reactor_pressure_rising"].sum()
        ),
        "boundary_fragments_retained": int(
            candidate_summary["boundary_fragment"].sum()
        ),
    },
    name="value",
).rename_axis("diagnostic").to_frame()

display(validation_report)
display(fragmentation_diagnostics)
display(candidate_summary.head(20))


## Interpretation limits

The construction returns half-open peak-to-peak cycles plus explicit leading and trailing fragments. Peak-event pressure is the appropriate property for ranking maxima; a within-cycle raw maximum can approach the next cycle's starting peak and must not be interpreted as that cycle's own peak. The objects remain pressure cycles, not validated fault diagnoses.